# P1 — FAISS Retrieval Mini-Project: Finding the Right Runbook Section
**GenAI Applications · graded (12%) · weeks 4–5 · individual**

## The story
**Chaos Corp** runs an online shop. Behind the website are about ten separate software components — one sends emails, one handles logins, one processes payments, and so on. Each of them is called a **service**.

Services break, also at night. So every night one engineer is **on call**: they carry a phone, and when something breaks, the monitoring system sends them an **alert**. Each such problem is called an **incident**. It is 03:00 and this alert arrives:

> `Ticket #9901 | Low Priority | Service: EmailNotifier — User complained about delayed reset password links. Logs show a 5-minute queue delay.`

Half asleep, the engineer has to answer three questions quickly:
1. *What is going on?* — is this a known kind of problem?
2. *What is safe to do now?* — which actions help without making things worse?
3. *Must I wake someone else up?* — e.g. the database team or the security team (this is called **escalating** or **paging** someone).

Nobody remembers the answers for ten different services. That is why companies write them down, service by service, as step-by-step instructions: what the service does, how its typical problems look, what to check first, what is safe to do, typical causes, and when to call whom. Such a document is called a **runbook** — literally "the book you run when something goes wrong". Chaos Corp has one runbook per service.

**The new problem:** there are many runbooks, plus other documents (old manuals, general guides, company policies), and they use similar words. At 03:00 the engineer needs **the right two or three passages**, not a pile of documents.

**This is exactly the problem RAG solves.** A language model alone does not know Chaos Corp's runbooks — and if you ask it anyway, it will invent a plausible answer. In **Retrieval-Augmented Generation (RAG)** (lectures: RAG architecture, embeddings, vector search, chunking) the system first **retrieves**: it searches a collection of documents (the *corpus*) for the specific pieces of information relevant to the question. Only then does the LLM **generate** an answer *from those pieces*.

- **P1 (this assignment) = the R:** given an incident log, find the runbook passages the engineer should read — and **measure** how well your retriever does it.
- **P2 = the G:** a model writes the triage (severity, cause, steps, whom to page) *using* what your retriever found.

A RAG system is only as good as its retriever: if the right passage is not found, the model cannot use it. That is why we measure retrieval carefully here.

## The plan
| Step | What you do | Lecture concept | Points |
|---|---|---|---|
| 0 | Setup | — | — |
| 1 | Get to know the documents, the incidents and the "right answers" | test collection, gold labels | — |
| 2 | ✏️ Cut documents into chunks (two strategies) | chunking | |
| 3 | ✏️ Embed chunks, search them with FAISS: your `retrieve()` | embeddings, vector search, metadata filter | **30** for working code (Steps 2–4) |
| 4 | ✏️ Measure retrieval quality: four metrics | hit@k, MRR, recall | |
| 5 | ✏️ Your own chunk size + compare 6 configurations and explain | controlled experiment | **15** own experiment + **20** answers |
| 6 | ✏️ Analyse your 5 weakest cases | error analysis | **30** |
| 7 | Clarity, reproducibility, submit | — | **5** clarity; −5 if it does not run |

**How you earn points:** working code gives 30 points — the ✅ Checks tell you when it is right. Most points (70) are for **your experiment, your evidence and your explanations**: answer with *your* numbers and *your* examples.

Cells marked **✏️ YOUR TURN** are your work; **✅ Check** cells test that your code has the right *structure* (they never reveal the gold answers). All other cells: run them and **read the output** — it explains the data.
**CPU track**

**Student:** Asliddin Tirkashev, RNU, Group 4230Z.

Execution environment: local runtime. The setup cells print the actual hardware, package versions and model settings. All 25 P1 incidents are evaluated on either track.


## Step 0 — Setup (run every session, ≈ 5 min)
Run the cells of Step 0 **in order**. You do not need to change anything except the settings cell.
If Colab disconnects, run Step 0 again; your finished results are skipped automatically.

### 0.1 Python packages
The completed run uses pinned versions listed below. In a fresh Colab runtime the cell installs these versions. Locally it verifies the installed environment. No forced kernel restart is needed.


In [1]:
import importlib.metadata, subprocess, sys
PACKAGES = {
    'requests': '2.32.5', 'numpy': '2.3.5', 'pandas': '2.2.3',
    'pydantic': '2.14.1', 'faiss-cpu': '1.15.1', 'tabulate': '0.10.0',
    'langchain': '1.0.2', 'langchain-ollama': '1.1.0'
}
if 'google.colab' in sys.modules:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q'] + [f'{k}=={v}' for k,v in PACKAGES.items()])
for package in PACKAGES:
    print(package, importlib.metadata.version(package))


requests 2.32.5
numpy 2.3.5
pandas 2.2.3
pydantic 2.14.1
faiss-cpu 1.15.1
tabulate 0.10.0
langchain 1.0.2
langchain-ollama 1.1.0


### 0.2 Settings — the only cell to edit in Steps 0.1–0.6
- `TRACK = "GPU"` if *Runtime → Change runtime type* shows **T4 GPU**; otherwise `"CPU"` (see *Getting started*).
- `USE_DRIVE = True` keeps your results on Google Drive (survives disconnects); `False` keeps everything in this session only.

In [2]:
import os, pathlib, importlib.util
try:
    IN_COLAB = importlib.util.find_spec('google.colab') is not None
except ModuleNotFoundError:
    IN_COLAB = False
TRACK = os.environ.get('GENAI_TRACK', 'CPU').upper()
os.environ['GENAI_TRACK'] = TRACK
USE_DRIVE = False
PRACTICE = 'P1'
MODEL = os.environ.get('GENAI_MODEL', 'qwen3:1.7b-q4_K_M')
OLLAMA_URL = os.environ.get('GENAI_OLLAMA_URL', 'http://localhost:11434')
os.environ['GENAI_OLLAMA_URL'] = OLLAMA_URL
OLLAMA_BIN = 'ollama'
CONTENT = pathlib.Path('/content') if IN_COLAB else pathlib.Path.cwd()
BASE = CONTENT
print('Track:', TRACK, '| model:', MODEL)


Track: CPU | model: qwen3:1.7b-q4_K_M


### 0.3 Check the runtime

In [3]:
import platform, shutil, subprocess
print('Runtime:', platform.platform())
if shutil.which('nvidia-smi'):
    print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], capture_output=True, text=True).stdout)
print('Selected track:', TRACK, '| num_gpu:', os.environ.get('GENAI_NUM_GPU', '99' if TRACK == 'GPU' else '0'))


Runtime: macOS-26.5.1-arm64-arm-64bit
Selected track: CPU | num_gpu: 0


### 0.4 Upload the data pack
A **"Choose Files"** button appears below the cell: select `chaoscorp_data_pack_v2.zip` (from Moodle). With `USE_DRIVE = True` a copy is kept on Drive and you will not be asked again.

In [4]:
import json, os, shutil, zipfile
DATA = pathlib.Path(os.environ.get('GENAI_DATA', str(CONTENT / 'chaoscorp')))
if not DATA.exists():
    for candidate in [CONTENT.parent / 'data/chaoscorp', CONTENT / 'data/chaoscorp']:
        if candidate.exists(): DATA = candidate; break
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = pathlib.Path('/content/drive/MyDrive/genai_course')
if not DATA.exists():
    from google.colab import files
    uploaded = files.upload()
    zip_path = next(iter(uploaded))
    with zipfile.ZipFile(zip_path) as z:
        assert 'chaoscorp/README.md' in z.namelist()
        z.extractall(CONTENT)
    DATA = CONTENT / 'chaoscorp'
print('Data directory:', DATA.name)
print('Data files:', sorted(p.name for p in DATA.iterdir()))


Data directory: chaoscorp
Data files: ['README.md', 'adversarial.json', 'incidents_dev.json', 'incidents_original10.json', 'incidents_test.json', 'p3_datasets', 'policies', 'runbooks', 'schema_v2.json', 'services.json']


### 0.5 Install and start Ollama (≈ 1–2 min)

In [5]:
import requests, subprocess, time, shutil
try:
    print('Ollama:', requests.get(OLLAMA_URL + '/api/version', timeout=10).json())
except requests.RequestException:
    if not shutil.which('ollama'):
        if not pathlib.Path('/content').exists(): raise RuntimeError('Install Ollama from its official site before running locally.')
        subprocess.check_call(['bash', '-lc', 'apt-get -qq install -y zstd pciutils > /dev/null && curl -fsSL https://ollama.com/install.sh | sh'])
    ollama = subprocess.Popen([OLLAMA_BIN, 'serve'], stdout=open(str(CONTENT / 'ollama.log'), 'w'), stderr=subprocess.STDOUT)
    for _ in range(60):
        try:
            requests.get(OLLAMA_URL + '/api/version', timeout=2).raise_for_status(); break
        except requests.RequestException: time.sleep(1)
    else: raise RuntimeError('Ollama did not start')


Ollama: {'version': '0.17.4'}


### 0.6 Helper functions (just run this cell)
You will use these helpers in every step. Read the one-line descriptions so you know what they do.

In [6]:
import json, os, time, requests

RESULTS = pathlib.Path(os.environ.get("GENAI_P1_RESULTS", str(BASE / "results" / PRACTICE if IN_COLAB or USE_DRIVE else CONTENT / "results")))
RESULTS.mkdir(parents=True, exist_ok=True)

def pull(model):
    """Download a model (skips if already present)."""
    r = requests.post(OLLAMA_URL + "/api/pull", json={"model": model, "stream": False}, timeout=3600)
    r.raise_for_status()
    return r.json().get("status")

def chat(messages, model=None, fmt=None, options=None):
    """One LLM call. Returns (answer_text, full_response). fmt: None, "json" or a JSON schema."""
    model = model or MODEL
    body = {"model": model, "messages": messages, "stream": False,
            "options": {"temperature": 0, "seed": 42, "num_ctx": 4096 if TRACK == "GPU" else 2048, "num_predict": 256 if TRACK == "GPU" else 200, "num_gpu": int(os.environ.get("GENAI_NUM_GPU", "99" if TRACK == "GPU" else "0")), **(options or {})}}
    if model.startswith("qwen3"):
        body["think"] = False          # no hidden reasoning tokens
    if fmt is not None:
        body["format"] = fmt
    r = requests.post(OLLAMA_URL + "/api/chat", json=body, timeout=900)
    r.raise_for_status()
    resp = r.json()
    return resp["message"]["content"], resp

def loaded_models():
    """What is in memory now, and which share of it is on the GPU (like `ollama ps`)."""
    out = []
    for m in requests.get(OLLAMA_URL + "/api/ps", timeout=10).json().get("models", []):
        size, vram = m.get("size", 0), m.get("size_vram", 0)
        out.append({"model": m["name"], "size_gb": round(size / 1e9, 2), "gpu_share": round(vram / size, 2) if size else 0.0})
    return out

def unload(model):
    """Remove a model from memory (like `ollama stop`)."""
    requests.post(OLLAMA_URL + "/api/generate", json={"model": model, "keep_alive": 0}, timeout=60)

def append_jsonl(path, record):
    """Save one finished result immediately (survives disconnects if USE_DRIVE)."""
    with open(path, "a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

def read_jsonl(path):
    if not os.path.exists(path):
        return []
    with open(path, encoding="utf-8") as f:
        return [json.loads(l) for l in f if l.strip()]

pull(MODEL)
text, resp = chat([{"role": "user", "content": "Say hello in five words."}])
print(text)
print("decode tok/s:", round(resp["eval_count"] / (resp["eval_duration"] / 1e9), 1))


Hello, world!
decode tok/s: 165.0


✅ **Check:** the cell above printed a short greeting and a speed. If it failed, run Step 0 again from 0.2.

### 0.7 ✏️ YOUR TURN 0 — use your model from Lab 1
P1 uses your model only for one small task: guessing the service name from the log (for the metadata filter in Step 3).
- If your Lab 1 results are on Google Drive, the cell below finds your model card automatically.
- Otherwise, type the model name from your Lab 1 model card (for example `"qwen3:4b-q4_K_M"`).

In [7]:
card_candidates = [BASE / 'results/L1/model_card.json', CONTENT.parent / 'l1/results/model_card.json']
for card in card_candidates:
    if card.exists():
        MODEL = json.loads(card.read_text())['model']; print('Model card found:', card.name); break
pull(MODEL)
print('This practice uses:', MODEL)
import hashlib
_files = sorted((DATA/'runbooks').glob('*.md')) + sorted((DATA/'policies').glob('*.md'))
_files += [DATA/name for name in ['incidents_test.json','services.json']]
_data_hash = hashlib.sha256(b''.join(p.name.encode()+p.read_bytes() for p in _files)).hexdigest()
_identity = {'model':MODEL,'embedding_model':'nomic-embed-text','data_sha256':_data_hash,
             'own_size':350,'own_overlap':60,'track':TRACK,'num_ctx':4096 if TRACK=='GPU' else 2048,
             'num_predict':256 if TRACK=='GPU' else 200,'seed':42,'temperature':0,
             'num_gpu':int(os.environ.get('GENAI_NUM_GPU','99' if TRACK=='GPU' else '0'))}
_identity_path = RESULTS/'experiment_identity.json'
if _identity_path.exists() and json.loads(_identity_path.read_text()) != _identity:
    raise ValueError('Saved results use a different model/data configuration. Select a fresh GENAI_P1_RESULTS directory.')
_identity_path.write_text(json.dumps(_identity,indent=2))


Model card found: model_card.json


This practice uses: qwen3:1.7b-q4_K_M


309

## Step 1 — The problem space: documents, incidents and the right answers
### 1.1 The document collection (the "corpus")
Our assistant may search **16 documents**. They are of three kinds:

| Kind | How many | What it is | Why it is here |
|---|---|---|---|
| **Service runbook** | 10 | How to handle incidents of one service (e.g. `AuthServer`, `EmailNotifier`) | the documents we *want* to find |
| **Distractor** | 3 | A **deprecated** payment gateway, a **reporting** batch job, a **generic** Kubernetes guide | they *sound* relevant (same words: payment, restart, pods) but are the wrong document — real document collections always contain such traps |
| **Policy** | 3 | Company-wide rules: severity guide, escalation policy, security policy | general rules that apply to every incident |

Run the cell: it loads all documents and prints this overview from the real files.

In [8]:
import re

def load_corpus():
    """All runbooks and policies as documents with their sections (character offsets in the document text)."""
    docs = []
    for path in sorted((DATA / "runbooks").glob("*.md")) + sorted((DATA / "policies").glob("*.md")):
        text = path.read_text()
        doc_id = re.search(r"^doc_id: (.+)$", text, re.M).group(1).strip()
        m = re.search(r"^service_name: (.+)$", text, re.M)
        service = m.group(1).strip() if m else "POLICY"
        heads = list(re.finditer(r"^## (s\d+_[a-z_]+) — (.+)$", text, re.M))
        sections = [{"section_id": h.group(1), "title": h.group(2), "start": h.start(),
                     "end": heads[i + 1].start() if i + 1 < len(heads) else len(text)} for i, h in enumerate(heads)]
        docs.append({"doc_id": doc_id, "service_name": service, "text": text, "sections": sections})
    return docs

def sections_overlapping(doc, start, end):
    """Section IDs of `doc` that overlap the character span [start, end)."""
    return [s["section_id"] for s in doc["sections"] if s["start"] < end and start < s["end"]]

DOCS = load_corpus()
DOC_BY_ID = {d["doc_id"]: d for d in DOCS}
DISTRACTORS = {"runbook_paymentgateway_v1_deprecated", "runbook_payment_reporting", "guide_platform_k8s_operations"}
SERVICE_NAMES = sorted({d["service_name"] for d in DOCS} - {"POLICY", "ALL", "PaymentGateway_v1", "payment-reporting"})
print(len(DOCS), "documents,", sum(len(d["sections"]) for d in DOCS), "sections")

import pandas as pd
def kind(d):
    return "distractor" if d["doc_id"] in DISTRACTORS else ("policy" if d["service_name"] == "POLICY" else "service runbook")
display(pd.DataFrame([{"doc_id": d["doc_id"], "kind": kind(d), "service": d["service_name"], "sections": len(d["sections"]),
                       "characters": len(d["text"])} for d in DOCS]).sort_values("kind"))

16 documents, 87 sections


,doc_id,kind,service,sections,characters
0,guide_platform_k8s_operations,distractor,ALL,6,4591
7,runbook_payment_reporting,distractor,payment-reporting,6,4609
8,runbook_paymentgateway_v1_deprecated,distractor,PaymentGateway_v1,6,4032
13,policy_escalation,policy,POLICY,3,1433
14,policy_security,policy,POLICY,3,1273
15,policy_severity,policy,POLICY,3,1292
1,runbook_authserver,service runbook,AuthServer,6,3691
2,runbook_database_cluster,service runbook,DatabaseCluster,6,3626
3,runbook_emailnotifier,service runbook,EmailNotifier,6,3419
4,runbook_frontend_react,service runbook,Frontend_React,6,3395


### 1.2 What is inside a runbook
Every service runbook (and every distractor) has the **same 6 sections**. An engineer uses them at different moments:

| Section | Contains | The engineer needs it to … |
|---|---|---|
| `s1_overview` | what the service does, its dependencies | understand the context |
| `s2_symptoms` | how incidents of this service look | **confirm** "yes, this is that kind of problem" |
| `s3_fast_triage` | first checks (dashboards, commands) | investigate |
| `s4_safe_mitigation` | safe actions that reduce the damage | **act** |
| `s5_root_causes` | typical causes and how to recognise them | find the cause |
| `s6_escalation` | when and whom to page (wake up) | **decide whether to escalate** |

Run the cell to read one real runbook, section by section (first lines of each).

In [9]:
doc = DOC_BY_ID["runbook_emailnotifier"]
for s in doc["sections"]:
    body = doc["text"][s["start"]:s["end"]].split("\n", 1)[1].strip()
    print(f"■ {s['section_id']}  ({s['end'] - s['start']} chars)\n   {body[:230]}...\n")

■ s1_overview  (574 chars)
   EmailNotifier, known as the email service, notifier, or mailer, sends transactional emails such as password resets, receipts, and notifications from a message queue. It depends on the external SMTP provider and the message queue. ...

■ s2_symptoms  (558 chars)
   Common symptoms are password-reset emails delayed, queue depth spikes, consumer lag measured in minutes, and SMTP 4xx/5xx responses from the external provider. Check the Mailer Queue Depth and Delivery Failure dashboards, then run...

■ s3_fast_triage  (568 chars)
   First establish whether the problem is queue-side or provider-side. Compare queue depth against the previous day and check consumer lag in minutes on the Mailer Lag panel. Run kubectl get pods -n mailer to identify missing, restar...

■ s4_safe_mitigation  (533 chars)
   If consumers are insufficient for load, scale them up by 1 or 2 temporarily with kubectl scale deployment/email-notifier -n mailer --replicas=4. Drain the backlog gr

### 1.3 The incidents (our queries) and the right answers
We have **25 test incidents**: real-looking logs written by engineers and monitoring tools — with typos, aliases ("mailer" instead of `EmailNotifier`), missing IDs, several services mentioned. Each log is a **query** for the retriever.

For each incident we know the **right answer** (the *gold label*, `gold.gold_sections`): the passages the engineer should read. The rule we used to create them follows the table above:
- always **`s2_symptoms`** (confirm) and **`s4_safe_mitigation`** (act) of the affected service's runbook,
- plus **`s6_escalation`** if the incident must be escalated to someone.

Gold sections are written as `doc_id#section_id`, e.g. `runbook_emailnotifier#s2_symptoms`. Gold labels are only for **evaluating** your retriever — never use them inside it. Run the cell to see three incidents and their right answers.

In [10]:
INCIDENTS = json.loads((DATA / "incidents_test.json").read_text())
print(len(INCIDENTS), "test incidents\n")
for inc in INCIDENTS[:3]:
    print("LOG:  ", inc["raw_log"].replace("\n", " / ")[:220])
    print("GOLD: ", inc["gold"]["service_name"], "->", inc["gold"]["gold_sections"])
    print()

25 test incidents

LOG:   2026-02-11T09:14:03Z alert | sev=LOW | service=checkout API | dubel charges detected for 42 orders after deploy; deploy diff shows PROVIDER_CAPTURE_MODE changed from 'single' to 'retry' / payment-processor acks show dupl
GOLD:  PaymentGateway_v2 -> ['runbook_paymentgateway_v2#s2_symptoms', 'runbook_paymentgateway_v2#s4_safe_mitigation', 'runbook_paymentgateway_v2#s6_escalation']

LOG:   2026-02-11T09:41:10Z event | service=PaymentGateway_v2 | internal synthetic checkout probes saw 502s for 38s after config push / auto rollback restored provider env; first successful synthetic charge at 09:41:48 / no cus
GOLD:  PaymentGateway_v2 -> ['runbook_paymentgateway_v2#s2_symptoms', 'runbook_paymentgateway_v2#s4_safe_mitigation']

LOG:   ALERT 2104 | Level: URGENT | mailer | SMTP provider outaage since 08:50, delay now 26m / queue depth 18k, customer receipts delayed / provider status page confirms regional outage
GOLD:  EmailNotifier -> ['runbook_emailnotifier#s2_sympto

Let us look at **what the gold passages actually say** for the first incident — this is exactly what a perfect retriever would bring to the engineer:

In [11]:
inc = INCIDENTS[0]
print("LOG:\n" + inc["raw_log"] + "\n")
for g in inc["gold"]["gold_sections"]:
    doc_id, sec_id = g.split("#")
    d = DOC_BY_ID[doc_id]; s = next(x for x in d["sections"] if x["section_id"] == sec_id)
    print(f"■ {g}\n{d['text'][s['start']:s['end']][:400]}...\n")

LOG:
2026-02-11T09:14:03Z alert | sev=LOW | service=checkout API | dubel charges detected for 42 orders after deploy; deploy diff shows PROVIDER_CAPTURE_MODE changed from 'single' to 'retry'
payment-processor acks show duplicate capture attempts; refunds pending
ops note: possible customer overbilling, please triage
no full outage, checkout responding

■ runbook_paymentgateway_v2#s2_symptoms
## s2_symptoms — Symptoms
Typical symptoms include HTTP 5xx on /checkout and /payments, customer reports of failed checkout, increased pod restarts, and elevated p95 latency on payment endpoints. Check the Payments Error Rate and Latency dashboards first, then inspect kubectl get events -n payments and kubectl top pods -n payments. Look for 502/504 spikes, AuthServer 401/403 bursts, payment-proces...

■ runbook_paymentgateway_v2#s4_safe_mitigation
## s4_safe_mitigation — Safe mitigation
If the error budget is burning, roll pods with kubectl rollout restart deployment/paymentgateway-v2 -n payments a

### 1.4 What we will build, and how we will know if it is good
```
incident log ──► embed (vector) ──► FAISS: most similar chunks ──► top-k chunks ──► compare with gold ──► metrics
                                         ▲
     documents ──► chunks ──► embed ─────┘   (done once: the index)
```
- **Why chunks?** We do not retrieve whole documents (too long for a prompt, and the engineer needs *one passage*), and not single sentences (too little context). We cut documents into pieces — **chunks** — and retrieve those. *How* to cut them is a design choice you will compare in Step 2.
- **When is a retrieved chunk "relevant"?** If it comes from the **gold runbook** and covers at least one **gold section**. Example for the EmailNotifier incident:
  - a chunk from `runbook_emailnotifier` covering `s4_safe_mitigation` → **relevant** ✅
  - a chunk from `runbook_emailnotifier` covering only `s1_overview` → not relevant (right document, wrong passage) ❌
  - a chunk from `runbook_payment_reporting` (a distractor) → not relevant ❌
- **Metrics** (Step 4) turn "relevant or not" for the top results into numbers, so we can compare designs objectively.

Two **provided helper functions** implement this rule; you will use them in Steps 4–6:
- `gold_set(inc)` turns an incident's gold sections (strings like `"runbook_emailnotifier#s2_symptoms"`) into a **set of pairs** `(doc_id, section_id)`, so they are easy to compare with chunks;
- `is_relevant(chunk, inc)` returns `True` if the chunk's document plus one of its sections is in that set.

Run the cell: it shows `gold_set` for one incident and checks the three examples above (expected: `True False False`).

In [12]:
def gold_set(inc):
    """Gold sections of an incident as a set of (doc_id, section_id) pairs."""
    return {tuple(s.split("#")) for s in inc["gold"]["gold_sections"]}

def is_relevant(chunk, inc):
    """True if the chunk comes from a gold document AND covers one of its gold sections."""
    return any((chunk["doc_id"], sec) in gold_set(inc) for sec in chunk["section_ids"])

inc = next(i for i in INCIDENTS if i["gold"]["service_name"] == "EmailNotifier")
print("gold_sections (strings):", inc["gold"]["gold_sections"])
print("gold_set (pairs):       ", gold_set(inc))
print("three examples:", is_relevant({"doc_id": "runbook_emailnotifier", "section_ids": ["s4_safe_mitigation"]}, inc),
      is_relevant({"doc_id": "runbook_emailnotifier", "section_ids": ["s1_overview"]}, inc),
      is_relevant({"doc_id": "runbook_payment_reporting", "section_ids": ["s4_safe_mitigation"]}, inc))

gold_sections (strings): ['runbook_emailnotifier#s2_symptoms', 'runbook_emailnotifier#s4_safe_mitigation', 'runbook_emailnotifier#s6_escalation']
gold_set (pairs):        {('runbook_emailnotifier', 's6_escalation'), ('runbook_emailnotifier', 's4_safe_mitigation'), ('runbook_emailnotifier', 's2_symptoms')}
three examples: True False False


## Step 2 — Chunking (part of the 30 code points)
**Why:** the retriever compares the *query* with each *chunk*. If a chunk mixes several topics, its embedding is a blurry average and it matches badly; if it is too small, it lacks context. We compare two common strategies:
- **C1 — fixed size:** every 500 characters, regardless of the content (simple, used by many tools by default);
- **C2 — structure-aware:** one chunk per section (uses the document's own structure).

A **chunk** is a dictionary:
```python
{"chunk_id": "runbook_authserver#C1-003", "doc_id": "runbook_authserver", "service_name": "AuthServer",
 "start": 1200, "end": 1700, "text": "...", "section_ids": ["s3_fast_triage", "s4_safe_mitigation"]}
```
`start`/`end` are character positions in the document text; `section_ids` says which sections the chunk covers (needed for the relevance rule). The helper `sections_overlapping(doc, start, end)` computes `section_ids` for you.

### ✏️ YOUR TURN 1 — C1: fixed-size chunks
Cut the document text into windows of `size` characters that **overlap** by `overlap` characters, ignoring section boundaries.
Fill in the three TODO lines of the loop:
1. the chunk ends at `start + size`, but never after the end of the text → `min(...)`;
2. append the chunk dictionary (template in the comment);
3. the next chunk starts `overlap` characters **before** the end of this one.

The loop stops after the chunk that reaches the end of the text (already written).

In [13]:
def _check_window(size, overlap):
    if not isinstance(size, int) or size <= 0 or not isinstance(overlap, int) or not 0 <= overlap < size:
        raise ValueError("size must be positive and overlap must be an integer in [0, size)")

def chunk_fixed(doc, size=500, overlap=50):
    _check_window(size, overlap)
    chunks, text, start = [], doc["text"], 0
    if not text:
        return chunks
    while True:
        end = min(start + size, len(text))
        chunks.append({"chunk_id": f"{doc['doc_id']}#C1-{len(chunks):03d}", "doc_id": doc["doc_id"],
                       "service_name": doc["service_name"], "start": start, "end": end,
                       "text": text[start:end], "section_ids": sections_overlapping(doc, start, end)})
        if end == len(text):
            break
        start = end - overlap
    return chunks


In [14]:
# ✅ Check (do not edit)
_t = {"doc_id": "t", "service_name": "X", "text": "a" * 1200,
      "sections": [{"section_id": "s1_a", "start": 0, "end": 600}, {"section_id": "s2_b", "start": 600, "end": 1200}]}
_c = chunk_fixed(_t, size=500, overlap=50)
assert len(_c) == 3, f"expected 3 chunks for 1200 chars (size 500, overlap 50), got {len(_c)}"
assert [(x["start"], x["end"]) for x in _c] == [(0, 500), (450, 950), (900, 1200)], "check your start/end positions"
assert _c[1]["section_ids"] == ["s1_a", "s2_b"], "the middle chunk overlaps both sections"
print("✅ chunk_fixed works.")

✅ chunk_fixed works.


### ✏️ YOUR TURN 2 — C2: structure-aware chunks
One chunk **per section**. If a section is longer than `max_len` characters, split *that section* into overlapping pieces: the same loop as C1, but it starts at `sec["start"]` and stops at `sec["end"]` instead of the end of the text.
Every C2 chunk has exactly **one** section ID: `[sec["section_id"]]`.

In [15]:
def chunk_by_section(doc, max_len=800, overlap=80):
    _check_window(max_len, overlap)
    chunks, text = [], doc["text"]
    for sec in doc["sections"]:
        start = sec["start"]
        while start < sec["end"]:
            end = min(start + max_len, sec["end"])
            chunks.append({"chunk_id": f"{doc['doc_id']}#C2-{len(chunks):03d}", "doc_id": doc["doc_id"],
                           "service_name": doc["service_name"], "start": start, "end": end,
                           "text": text[start:end], "section_ids": [sec["section_id"]]})
            if end == sec["end"]:
                break
            start = end - overlap
    return chunks


In [16]:
# ✅ Check (do not edit)
_t = {"doc_id": "t", "service_name": "X", "text": "b" * 1300,
      "sections": [{"section_id": "s1_a", "start": 0, "end": 300}, {"section_id": "s2_b", "start": 300, "end": 1300}]}
_c = chunk_by_section(_t, max_len=800, overlap=80)
assert _c and all(len(x["section_ids"]) == 1 for x in _c), "every C2 chunk must have exactly one section id"
assert _c[0]["start"] == 0 and _c[0]["end"] == 300, "a short section is one chunk"
assert sum(1 for x in _c if x["section_ids"] == ["s2_b"]) == 2, "a 1000-char section with max_len 800 becomes 2 pieces"
assert all(x["end"] - x["start"] <= 800 for x in _c), "no chunk may be longer than max_len"
print("✅ chunk_by_section works.")

✅ chunk_by_section works.


### Build both chunk sets and look at the statistics (just run)

In [17]:
import pandas as pd
CHUNKS = {"C1": [ch for d in DOCS for ch in chunk_fixed(d)], "C2": [ch for d in DOCS for ch in chunk_by_section(d)]}
stats = pd.DataFrame([{"strategy": k, "chunks": len(v), "mean_len": round(sum(len(x["text"]) for x in v) / len(v)),
                       "min_len": min(len(x["text"]) for x in v), "max_len": max(len(x["text"]) for x in v),
                       "chunks spanning >1 section": sum(len(x["section_ids"]) > 1 for x in v)} for k, v in CHUNKS.items()])
display(stats)

,strategy,chunks,mean_len,min_len,max_len,chunks spanning >1 section
0,C1,123,465,61,500,78
1,C2,90,563,87,800,0


## Step 3 — Index and retrieve (part of the 30 code points)
**Why:** an **embedding model** (`nomic-embed-text`) turns a text into a vector (a list of numbers) so that texts with similar *meaning* get similar vectors — even with different words ("mailer delay" ≈ "email queue lag"). We embed every chunk once and store the vectors in a **FAISS index**; for an incident we embed the log and ask FAISS for the most similar chunk vectors (cosine similarity).
The embedding and index code is **provided** (run it). Your part is the search function with an optional **metadata filter**: if we already know the service, we can search only that service's runbook (plus the policies) and so never return a distractor.

In [18]:
import hashlib, urllib.request
import numpy as np, faiss
from pathlib import Path
EMBED_MODEL = 'nomic-embed-text'
pull(EMBED_MODEL)

def api_post(route, payload, timeout=900):
    url = os.environ.get("GENAI_OLLAMA_URL", "http://localhost:11434")
    req = urllib.request.Request(url + route, json.dumps(payload).encode(), {"Content-Type": "application/json"})
    with urllib.request.urlopen(req, timeout=timeout) as r:
        return json.load(r)

class Embedder:
    """Persistent document cache; query timings include actual uncached API embedding."""
    def __init__(self, cache_dir, model=EMBED_MODEL):
        self.cache_dir = Path(cache_dir)
        self.cache_dir.mkdir(parents=True, exist_ok=True)
        self.model = model

    def embed(self, texts, kind, cache=True):
        if kind not in {"document", "query"}:
            raise ValueError("kind must be document or query")
        prefix = "search_document: " if kind == "document" else "search_query: "
        keys = [hashlib.sha256((self.model + "\0" + prefix + text).encode()).hexdigest() for text in texts]
        vectors = [None] * len(texts)
        missing = []
        for i, key in enumerate(keys):
            file = self.cache_dir / (key + ".npy")
            if cache and file.exists():
                vectors[i] = np.load(file, allow_pickle=False)
            else:
                missing.append(i)
        for start in range(0, len(missing), 32):
            batch = missing[start:start + 32]
            resp = api_post("/api/embed", {"model": self.model, "input": [prefix + texts[i] for i in batch],
                                            "options": {"num_gpu": int(os.environ.get("GENAI_NUM_GPU", "99" if os.environ.get("GENAI_TRACK", "CPU").upper() == "GPU" else "0"))}}, timeout=900)
            if len(resp["embeddings"]) != len(batch):
                raise ValueError("Embedding API returned wrong vector count")
            for i, vector in zip(batch, resp["embeddings"]):
                v = np.asarray(vector, dtype="float32")
                norm = np.linalg.norm(v)
                if not np.isfinite(norm) or norm <= 0:
                    raise ValueError("Invalid embedding vector")
                vectors[i] = v / norm
                if cache:
                    np.save(self.cache_dir / (keys[i] + ".npy"), vectors[i], allow_pickle=False)
        result = np.asarray(vectors, dtype="float32")
        faiss.normalize_L2(result)
        return result

EMBEDDER = Embedder(RESULTS / 'embedding_cache')
def embed(texts, kind):
    return EMBEDDER.embed(texts, kind, cache=kind == 'document')
def build_index(chunks):
    vecs = embed([c['text'] for c in chunks], 'document')
    index = faiss.IndexFlatIP(vecs.shape[1]); index.add(vecs)
    return {'index': index, 'chunks': chunks}
INDEXES = {name: build_index(chs) for name, chs in CHUNKS.items()}
print({k: v['index'].ntotal for k,v in INDEXES.items()}, 'vectors indexed')


{'C1': 123, 'C2': 90} vectors indexed


### ✏️ YOUR TURN 3 — `retrieve()`
Return the `k` most similar chunks for a query, optionally **only from one service's runbook plus the 3 policies**.
Steps:
1. embed the query: `q = embed([query_text], "query")`;
2. search all candidates for exact metadata-filtered top-k, because the filter will remove some: `scores, ids = idx["index"].search(q, idx["index"].ntotal)`;
3. go through `ids[0]` / `scores[0]` in order; skip chunks that do not pass the filter (when `service_filter` is set and not `"UNKNOWN"`: keep only chunks whose `service_name == service_filter` or `== "POLICY"`);
4. return the first `k` as dicts: the chunk's fields plus `"score": float(score)`.


In [19]:
def retrieve_vector(q, idx, k=5, service_filter=None):
    if k <= 0:
        return []
    # Search all candidates, then retain the permitted metadata subset. This is
    # exact filtered top-k; a fixed shortlist could discard permitted candidates.
    scores, ids = idx["index"].search(q, idx["index"].ntotal)
    results = []
    filtering = bool(service_filter) and service_filter != "UNKNOWN"
    for score, i in zip(scores[0], ids[0]):
        if i < 0:
            continue
        ch = idx["chunks"][int(i)]
        if filtering and ch["service_name"] not in {service_filter, "POLICY"}:
            continue
        results.append({**ch, "score": float(score)})
        if len(results) == k:
            break
    return results

def retrieve(query_text, idx, k=5, service_filter=None):
    q = embed([query_text], 'query')
    return retrieve_vector(q, idx, k, service_filter)


In [20]:
# ✅ Check (do not edit)
_r = retrieve("Login is slow and auth pods were OOMKilled", INDEXES["C2"], k=5)
assert len(_r) == 5 and all("score" in x and "section_ids" in x for x in _r), "return k dicts with the chunk fields and a score"
assert all(_r[i]["score"] >= _r[i + 1]["score"] for i in range(4)), "results must be sorted by score (highest first)"
_f = retrieve("Login is slow and auth pods were OOMKilled", INDEXES["C2"], k=5, service_filter="EmailNotifier")
assert all(x["service_name"] in ("EmailNotifier", "POLICY") for x in _f), "with a filter, only that service's runbook + policies"
print("✅ retrieve works. Top hit:", _r[0]["chunk_id"], round(_r[0]["score"], 3))

✅ retrieve works. Top hit: runbook_authserver#C2-002 0.704


### The service guess for the filter (provided)
The filter needs the service name. In real life we do not have the gold label, so we **guess it from the log with your model** (a small Lab-2-style extractor). Never use `gold.service_name` for the filter: that would be data leakage.

In [21]:
SERVICES = json.loads((DATA / 'services.json').read_text())['services']
DEFAULT_MODEL = MODEL

def extract_service(raw_log, services, model=DEFAULT_MODEL, log_path=None):
    names = [s["service_name"] for s in services]
    schema = {"type": "object", "properties": {"service_name": {"type": "string", "enum": names + ["UNKNOWN"]}},
              "required": ["service_name"], "additionalProperties": False}
    aliases = "\n".join(f"- {s['service_name']} (also called: {', '.join(s['aliases'])})" for s in services)
    messages = [
        {"role": "system", "content": "Name the affected Chaos Corp service. Map aliases to the canonical name. "
         "If two services are mentioned, choose the one where the fault is.\n" + aliases},
        {"role": "user", "content": raw_log},
    ]
    body = {"model": model, "messages": messages, "stream": False, "format": schema,
            "options": {"temperature": 0, "seed": 42,
                        "num_ctx": 4096 if os.environ.get("GENAI_TRACK", "CPU").upper() == "GPU" else 2048,
                        "num_predict": 256 if os.environ.get("GENAI_TRACK", "CPU").upper() == "GPU" else 200,
                        "num_gpu": int(os.environ.get("GENAI_NUM_GPU", "99" if os.environ.get("GENAI_TRACK", "CPU").upper() == "GPU" else "0"))}}
    if model.startswith("qwen3"):
        body["think"] = False
    t0 = time.perf_counter()
    resp = api_post("/api/chat", body)
    latency = time.perf_counter() - t0
    raw = resp["message"]["content"]
    try:
        service = json.loads(raw)["service_name"]
        if service not in names + ["UNKNOWN"]:
            raise ValueError("unknown canonical service")
        error = None
    except (ValueError, KeyError, TypeError) as e:
        service, error = "UNKNOWN", str(e)
    if log_path:
        append_jsonl(log_path, {"query_sha256": hashlib.sha256(raw_log.encode()).hexdigest(), "model": model,
                                "raw_log": raw_log, "messages": messages, "raw_output": raw,
                                "service_name": service, "latency_s": latency, "error": error,
                                "eval_count": resp.get("eval_count"), "eval_duration": resp.get("eval_duration")})
    return service

def service_for_log(raw_log):
    return extract_service(raw_log, SERVICES, MODEL, RESULTS / 'service_calls.jsonl')
prediction_path = RESULTS / 'service_predictions.jsonl'
_saved_predictions = {r['incident_key']:r for r in read_jsonl(prediction_path)}
GUESSED = {}
for inc in INCIDENTS:
    key = inc['incident_key']
    if key not in _saved_predictions:
        t0 = time.perf_counter(); service = service_for_log(inc['raw_log'])
        record = {'incident_key':key, 'service_name':service, 'latency_s':time.perf_counter()-t0}
        append_jsonl(prediction_path, record); _saved_predictions[key] = record
    GUESSED[key] = _saved_predictions[key]['service_name']
correct = sum(GUESSED[i['incident_key']] == i['gold']['service_name'] for i in INCIDENTS)
print(f'Service guessed correctly for {correct}/{len(INCIDENTS)} incidents')
print('Persisted predictions are reused when resuming this exact model/data configuration.')


Service guessed correctly for 24/25 incidents
Persisted predictions are reused when resuming this exact model/data configuration.


## Step 4 — Metrics (part of the 30 code points)
**Why:** "it looks good on two examples" is not evidence. Metrics summarise, over all 25 incidents, *how often* and *how high* the relevant passages appear — so you can compare designs objectively:
- **hit@k** — did the engineer get *at least one* useful passage among the first k? (hit@1: the very first result)
- **MRR** — *how high* is the first useful passage? (1st place = 1, 2nd = 0.5, 3rd = 0.33 …)
- **section recall** — did the top 5 cover *all* the passages the engineer needs (symptoms, mitigation, escalation)?
- **distractor@3** — how often does a trap document appear in the top 3?

For one incident, `rel` is the list of relevance flags of the retrieved chunks in rank order, e.g. `[False, True, False, False, False]` (the 2nd chunk is relevant).

### ✏️ YOUR TURN 4 — implement the four metrics
| Metric | Definition |
|---|---|
| `hit_at_k(rel, k)` | 1 if any of the first `k` flags is True, else 0 |
| `mrr(rel, k=5)` | 1 / (rank of the first True within the first `k`), ranks start at 1; 0 if none |
| `section_recall(chunks, inc, k=5)` | share of the incident's gold sections covered by the `section_ids` of the first `k` chunks of the gold runbook |
| `distractor_at_3(chunks)` | 1 if one of the first 3 chunks comes from a distractor document (`DISTRACTORS`), else 0 |

In [22]:
def hit_at_k(rel, k):
    return int(any(rel[:k]))

def mrr(rel, k=5):
    return next((1.0 / (i + 1) for i, relevant in enumerate(rel[:k]) if relevant), 0.0)

def section_recall(chunks, inc, k=5):
    gold = gold_set(inc)
    covered = {(chunk["doc_id"], sec) for chunk in chunks[:k] for sec in chunk["section_ids"]}
    return len(gold & covered) / len(gold) if gold else 1.0

def distractor_at_3(chunks):
    return int(any(c["doc_id"] in DISTRACTORS for c in chunks[:3]))


In [23]:
# ✅ Check (do not edit)
assert hit_at_k([False, True, False], 1) == 0 and hit_at_k([False, True, False], 3) == 1
assert abs(mrr([False, False, True, False, False]) - 1 / 3) < 1e-9 and mrr([False] * 5) == 0
_inc = {"gold": {"gold_sections": ["d#s2_symptoms", "d#s4_safe_mitigation"]}}
_ch = [{"doc_id": "d", "section_ids": ["s2_symptoms"]}, {"doc_id": "x", "section_ids": ["s4_safe_mitigation"]}]
assert section_recall(_ch, _inc) == 0.5, "only sections of the gold runbook count"
assert distractor_at_3([{"doc_id": "a"}, {"doc_id": "runbook_payment_reporting"}]) == 1
print("✅ All four metrics work.")

✅ All four metrics work.


## Step 5 — Your experiment and the comparison (15 + 20 pts)
### ✏️ YOUR TURN 5 — your own chunk size (15 pts)
C1 uses 500 characters with 50 overlap — but is that a good choice? Pick **your own** size and overlap (different from 500 / 50), write down what you **expect** before running, and the table below will include your version as `C1_own`. Every student's choice is different, so your results are your own.
Ideas: much smaller chunks (e.g. 250) are more precise but have less context; much larger ones (e.g. 1000) cover more sections at once.

In [24]:
MY_SIZE = 350
MY_OVERLAP = 60
MY_EXPECTATION = 'I expect 350-character chunks with 60-character overlap to improve first-hit precision over 500/50 by mixing fewer sections, while possibly reducing section recall because each retrieved chunk covers less text.'
print('Advance expectation:', MY_EXPECTATION)
prereg = RESULTS / 'preregistered_experiment.json'
if not prereg.exists():
    prereg.write_text(json.dumps({'size':MY_SIZE, 'overlap':MY_OVERLAP, 'expectation':MY_EXPECTATION, 'recorded_before_run':time.strftime('%Y-%m-%dT%H:%M:%SZ',time.gmtime())}, indent=2))
assert 0 <= MY_OVERLAP < MY_SIZE and (MY_SIZE, MY_OVERLAP) != (500,50)
CHUNKS['C1_own'] = [ch for d in DOCS for ch in chunk_fixed(d, MY_SIZE, MY_OVERLAP)]
INDEXES['C1_own'] = build_index(CHUNKS['C1_own'])
print('C1_own chunks:', len(CHUNKS['C1_own']))


Advance expectation: I expect 350-character chunks with 60-character overlap to improve first-hit precision over 500/50 by mixing fewer sections, while possibly reducing section recall because each retrieved chunk covers less text.
C1_own chunks: 184


### The comparison: 3 chunkings × {no filter, service filter} (just run; ≈ 1–2 min)

In [25]:
_retrieval_path = RESULTS / 'retrieval_results.jsonl'
_saved_rows = {(r['chunking'],r['use_filter'],r['incident_key']):r for r in read_jsonl(_retrieval_path)}
def evaluate_config(chunking, use_filter):
    rows = []
    for inc in INCIDENTS:
        key = (chunking,use_filter,inc['incident_key'])
        if key in _saved_rows:
            rows.append(_saved_rows[key]); continue
        t0 = time.perf_counter()
        svc = GUESSED[inc['incident_key']] if use_filter else None
        chunks = retrieve(inc['raw_log'], INDEXES[chunking], 5, svc)
        latency_ms = 1000*(time.perf_counter()-t0)
        rel = [is_relevant(ch,inc) for ch in chunks]
        row = {'chunking':chunking,'use_filter':use_filter,'incident_key':inc['incident_key'],
               'guessed_service':GUESSED[inc['incident_key']], 'hit@1':hit_at_k(rel,1),'hit@3':hit_at_k(rel,3),
               'mrr@5':mrr(rel),'section_recall@5':section_recall(chunks,inc),'distractor@3':distractor_at_3(chunks),
               'latency_ms':latency_ms,'retrieved':chunks,
               'service_extraction_ms':1000*_saved_predictions[inc['incident_key']]['latency_s'] if use_filter else 0,
               'retrieved_characters@5':sum(len(ch['text']) for ch in chunks)}
        append_jsonl(_retrieval_path,row); _saved_rows[key] = row; rows.append(row)
    return pd.DataFrame(rows)
CONFIGS = {(ch,f):evaluate_config(ch,f) for ch in ('C1','C2','C1_own') for f in (False,True)}
metric_names = ['hit@1','hit@3','mrr@5','section_recall@5','distractor@3','latency_ms','service_extraction_ms','retrieved_characters@5']
summary = pd.DataFrame([{'chunking':ch,'filter':f,**frame[metric_names].mean().to_dict()} for (ch,f),frame in CONFIGS.items()])
display(summary.round(3))
summary.to_csv(RESULTS/'p1_notebook_summary.csv',index=False)
print('Latency: query embedding plus exact FAISS search. Extractor cost is shown separately.')


,chunking,filter,hit@1,hit@3,mrr@5,section_recall@5,distractor@3,latency_ms,service_extraction_ms,retrieved_characters@5
0,C1,False,0.56,0.88,0.733,0.760,0.20,67.153,0.000,2478.76
1,C1,True,0.72,0.96,0.833,0.893,0.00,34.994,958.736,2489.12
2,C2,False,0.36,0.72,0.538,0.487,0.12,36.076,0.000,2845.20
3,C2,True,0.36,0.80,0.587,0.687,0.00,35.783,958.736,2790.04
4,C1_own,False,0.36,0.92,0.620,0.627,0.08,37.687,0.000,1735.12
5,C1_own,True,0.60,0.96,0.773,0.847,0.00,35.407,958.736,1738.64


Latency: query embedding plus exact FAISS search. Extractor cost is shown separately.


### Where did the filter hurt? (incidents where the filtered config is worse at hit@3)

In [26]:
best_chunking = summary.sort_values("mrr@5", ascending=False, kind="stable").iloc[0]["chunking"]
a, b = CONFIGS[(best_chunking, False)], CONFIGS[(best_chunking, True)]
hurt = a[(a["hit@3"] == 1) & (b["hit@3"] == 0)]["incident_key"]
gold_by_key = {i["incident_key"]: i["gold"] for i in INCIDENTS}
display(pd.DataFrame([{"incident_key": k, "guessed service": GUESSED[k], "gold service": gold_by_key[k]["service_name"],
                       "tags": ", ".join(gold_by_key[k]["hard_case_tags"])} for k in hurt]) if len(hurt) else "The filter never hurt hit@3.")


,incident_key,guessed service,gold service,tags
0,NEW-2-09,PaymentGateway_v2,payment-processor,"H1_missing_field, H4_ambiguous_service"


### ✏️ YOUR TURN 6 — answer with your numbers (20 pts, ≈ 200 words)
1. Which single change helped more: the **chunking** (C1 → C2) or the **filter**? Quote the numbers.
2. Did the filter ever **hurt**? Look at the table above: was the guessed service wrong? Which hard-case tags do these incidents have?
3. Look at the column `chunks spanning >1 section` in the Step 2 statistics. A C1 chunk that spans two sections counts as relevant if *either* section is a gold section. Could that make C1 look better than it really is? What would a fair comparison need (hint: how much text does the user have to read)?
4. **Your own chunk size:** was your expectation right? What changed compared with C1 (500), and what explains it?

I compared six configurations on the same 25 incident logs. Without filtering, changing C1 to C2 changed MRR@5 from 0.733 to 0.538 (-0.195). Keeping C1 and enabling the predicted-service filter changed it to 0.833 (+0.101). The filter therefore produced the larger MRR change. Filtering lost hit@3 for NEW-2-09 (H1_missing_field/H4_ambiguous_service). For NEW-2-09, the guessed service was PaymentGateway_v2 and the gold service was payment-processor. I use model-predicted services, never gold services, for filtering. A wrong prediction can remove the correct runbook entirely.

My 350/60 chunks gave MRR@5 0.620 and section recall@5 0.627, compared with 0.733 and 0.760 for 500/50. Hit@1 fell from 0.560 to 0.360, so my expected precision improvement did not appear. The anticipated coverage cost appeared. C1 has 78 chunks crossing section boundaries. Under the supplied relevance rule, touching either gold section is enough to count as relevant, even when much of the chunk is unrelated. I would also compare quality at a fixed retrieved-character budget and record the amount of text an engineer must read.

The latency table measures query embedding plus exact vector search. Service extraction is a separate model call, reported separately rather than hidden inside the filter comparison. The runtime and selected track are recorded above and in runtime.json.


## Step 6 — Failure analysis (30 pts)
Even a good retriever has **weak cases**. A case can be weak in three ways (from bad to less bad):
1. **no relevant passage in the top 3** (hit@3 = 0) — the engineer finds nothing useful;
2. the relevant passage is there but **not first** (hit@1 = 0, MRR < 1) — the engineer reads wrong text first;
3. **some needed sections are missing** from the top 5 (section recall < 1), e.g. symptoms are found but not the escalation rules.

The cell ranks all incidents of your **best configuration** and shows the **5 weakest** with the reason. Use `show_failure(...)` to look at each one.

In [27]:
best = summary.sort_values('mrr@5', ascending=False, kind='stable').iloc[0]
BEST = (best['chunking'], bool(best['filter']))
ranked = CONFIGS[BEST].sort_values(['hit@3','mrr@5','section_recall@5'], kind='stable').copy()
def _why(r):
    if r['hit@3'] == 0: return 'no relevant passage in top 3'
    if r['hit@1'] == 0: return 'relevant passage not first'
    if r['section_recall@5'] < 1: return 'some gold sections missing from top 5'
    return 'perfect'
ranked['why weak'] = ranked.apply(_why,axis=1)
WEAKEST = ranked.head(5)
print('Best configuration:', BEST)
display(WEAKEST[['incident_key','hit@1','hit@3','mrr@5','section_recall@5','why weak']])
fails = WEAKEST['incident_key'].tolist()
def show_failure(incident_key):
    inc = next(i for i in INCIDENTS if i['incident_key']==incident_key)
    row = CONFIGS[BEST].set_index('incident_key').loc[incident_key]
    print('LOG:',inc['raw_log'])
    print('GOLD:',inc['gold']['gold_sections'],'| tags:',inc['gold']['hard_case_tags'],'| guessed:',GUESSED[incident_key])
    for x in row['retrieved'][:3]: print(f"{x['score']:.4f} {x['doc_id']} {x['section_ids']} | {x['text']}")


Best configuration: ('C1', True)


,incident_key,hit@1,hit@3,mrr@5,section_recall@5,why weak
18,NEW-2-09,0,0,0.000000,0.000000,no relevant passage in top 3
7,NEW-1-11,0,1,0.333333,1.000000,relevant passage not first
22,NEW-2-13,0,1,0.500000,0.666667,relevant passage not first
1,NEW-1-03,0,1,0.500000,1.000000,relevant passage not first
5,NEW-1-09,0,1,0.500000,1.000000,relevant passage not first


## Five weakest cases: diagnosis

| Case | Incident | Top-3 documents / sections / scores | Gold sections | Diagnosis | Explanation |
|---|---|---|---|---|---|
| 1 | NEW-2-09 | runbook_paymentgateway_v2 / s6_escalation / 0.674<br>runbook_paymentgateway_v2 / s1_overview,s2_symptoms / 0.673<br>policy_severity / s3_no_label / 0.670 | runbook_payment_processor#s2_symptoms<br>runbook_payment_processor#s4_safe_mitigation | alias misclassification and wrong filter | The log says “payments worker”, a listed payment-processor alias. The model chose PaymentGateway_v2, so filtering excluded every gold chunk. |
| 2 | NEW-1-11 | runbook_database_cluster / s3_fast_triage / 0.698<br>runbook_database_cluster / s5_root_causes,s6_escalation / 0.691<br>runbook_database_cluster / s4_safe_mitigation,s5_root_causes / 0.686 | runbook_database_cluster#s2_symptoms<br>runbook_database_cluster#s4_safe_mitigation | diagnostic text outranks labelled action sections | Lock-related wording ranks fast triage and root causes first. Gold mitigation is third and symptoms fourth, so coverage is complete but MRR is 1/3. |
| 3 | NEW-2-13 | runbook_recommendations_ai / s5_root_causes / 0.724<br>runbook_recommendations_ai / s4_safe_mitigation,s5_root_causes / 0.708<br>runbook_recommendations_ai / s1_overview / 0.686 | runbook_recommendations_ai#s2_symptoms<br>runbook_recommendations_ai#s4_safe_mitigation<br>runbook_recommendations_ai#s6_escalation | escalation section missing | GPU saturation closely matches the root-cause chunk. Mitigation ranks second and symptoms fourth, but none of the five chunks covers escalation. |
| 4 | NEW-1-03 | runbook_paymentgateway_v2 / s5_root_causes,s6_escalation / 0.719<br>runbook_paymentgateway_v2 / s4_safe_mitigation,s5_root_causes / 0.701<br>runbook_paymentgateway_v2 / s6_escalation / 0.689 | runbook_paymentgateway_v2#s2_symptoms<br>runbook_paymentgateway_v2#s4_safe_mitigation | cause and escalation outrank labelled symptoms | Provider settings and rollback wording match the mixed cause/escalation chunk first. Gold mitigation is second and symptoms fifth, giving complete coverage but MRR 1/2. |
| 5 | NEW-1-09 | runbook_authserver / s5_root_causes,s6_escalation / 0.733<br>runbook_authserver / s4_safe_mitigation,s5_root_causes / 0.716<br>runbook_authserver / s1_overview,s2_symptoms / 0.688 | runbook_authserver#s2_symptoms<br>runbook_authserver#s4_safe_mitigation | cause description outranks labelled mitigation | The phrase “signing-key rotation” matches the root-cause chunk first. Gold mitigation is second and symptoms third, giving complete coverage but MRR 1/2. |

**Tested fix:** I prefixed every query with the same request for symptoms, safe mitigation and escalation. Section recall rose from 0.893 to 0.913, but MRR fell from 0.833 to 0.767. I keep the original best configuration. This exploratory result reuses the inspected 25 logs and needs held-out validation.

The code cell below displays the complete query and top-three source passages for each case, followed by the separate exploratory comparison.


In [28]:
for key in fails:
    print('\n' + '='*80)
    show_failure(key)

# Separate exploratory fix. The six required configurations above are unchanged.
EXPANSION = 'Find symptoms, safe mitigation and escalation for this incident:\n'
fix_plan = RESULTS/'query_expansion_plan.json'
if not fix_plan.exists():
    fix_plan.write_text(json.dumps({'prefix':EXPANSION,
        'expectation':'I expect higher escalation coverage, with possible loss of first-hit ranking. A wrong service filter remains a problem.',
        'recorded_before_run':time.strftime('%Y-%m-%dT%H:%M:%SZ',time.gmtime())},indent=2))
assert json.loads(fix_plan.read_text())['prefix']==EXPANSION
fix_path = RESULTS/'query_expansion_results.jsonl'
fix_rows = {r['incident_key']:r for r in read_jsonl(fix_path)}
for inc in INCIDENTS:
    key=inc['incident_key']
    if key in fix_rows: continue
    t0=time.perf_counter()
    chunks=retrieve(EXPANSION+inc['raw_log'],INDEXES[BEST[0]],5,GUESSED[key] if BEST[1] else None)
    elapsed=1000*(time.perf_counter()-t0)
    rel=[is_relevant(c,inc) for c in chunks]
    row={'incident_key':key,'hit@1':hit_at_k(rel,1),'hit@3':hit_at_k(rel,3),'mrr@5':mrr(rel),
         'section_recall@5':section_recall(chunks,inc),'distractor@3':distractor_at_3(chunks),
         'latency_ms':elapsed,'guessed_service':GUESSED[key],'retrieved':chunks}
    append_jsonl(fix_path,row);fix_rows[key]=row
print('Exploratory query expansion, same inspected incidents:')
display(pd.DataFrame(fix_rows.values())[['hit@1','hit@3','mrr@5','section_recall@5','distractor@3','latency_ms']].mean().to_frame('mean'))
print('Recorded query times may include model loading and endpoint scheduling. Service extraction is reported separately.')



LOG: payments worker note
low priority
3 sandbox transactions missing idempotency keys; no production customer impact.
Closed by QA after replay test.
GOLD: ['runbook_payment_processor#s2_symptoms', 'runbook_payment_processor#s4_safe_mitigation'] | tags: ['H1_missing_field', 'H4_ambiguous_service'] | guessed: PaymentGateway_v2
0.6744 runbook_paymentgateway_v2 ['s6_escalation'] | # s6_escalation — Escalation
If this is high severity or checkout has been down for more than 5 minutes, page SRE_ONCALL. Attach the Payments Overview screenshot, affected /checkout and /payments paths, request IDs, deploy timestamp, and current mitigation state. If there is suspected financial impact, such as double charges or lost payments, page SRE_ONCALL and FINOPS. Include customer-visible failure estimates, transaction identifiers where available, and evidence that retries did not resolve 
0.6729 runbook_paymentgateway_v2 ['s1_overview', 's2_symptoms'] | for /checkout and /payments, p95 latency, pod rest

,mean
hit@1,0.600000
hit@3,0.960000
mrr@5,0.766667
section_recall@5,0.913333
distractor@3,0.000000
latency_ms,331.027143


Recorded query times may include model loading and endpoint scheduling. Service extraction is reported separately.


## Step 7 — Reproducibility (just run) and submit
**Clarity (5 pts):** your tables are visible, every answer cell is filled in, and the GPU/CPU track is written at the top.

In [29]:
import sys, importlib.metadata, platform
print('Python:',sys.version.split()[0], '| platform:',platform.platform())
for name in ['numpy','pandas','faiss-cpu','requests','pydantic','langchain','langchain-ollama']:
    print(name,importlib.metadata.version(name))
print('Models:', MODEL, EMBED_MODEL, '| track:',TRACK, '| num_gpu:',os.environ.get('GENAI_NUM_GPU','99' if TRACK=='GPU' else '0'), '| seed:42 | temperature:0')


Python: 3.12.14 | platform: macOS-26.5.1-arm64-arm-64bit
numpy 2.3.5
pandas 2.2.3
faiss-cpu 1.15.1
requests 2.32.5
pydantic 2.14.1
langchain 1.0.2
langchain-ollama 1.1.0
Models: qwen3:1.7b-q4_K_M nomic-embed-text | track: CPU | num_gpu: 0 | seed:42 | temperature:0


**Submit:** *File → Download → Download .ipynb* and upload it to Moodle. Check that all tables, your answers and the failure table are visible.
Discussion length overall ≈ 300 words (Step 5 + Step 6).